# 04 — Topic modeling với TF-IDF + NMF

Fit mô hình trên mẫu tối đa 50.000 bài để kiểm soát RAM, sau đó gán chủ đề cho toàn bộ dữ liệu sentiment theo từng batch. Kết quả là chủ đề tự động khám phá, cần đọc mẫu đại diện và đặt tên thủ công.

In [ ]:
%pip -q install pandas pyarrow scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/RedditProject')
DATA_DIR = PROJECT_DIR / 'data/processed'
SCORED_PATH = DATA_DIR / 'reddit_posts_sentiment.parquet'
assert SCORED_PATH.exists(), f'Không thấy {SCORED_PATH}; chạy 03_sentiment_trends.ipynb trước.'
N_TOPICS = 12
FIT_SAMPLE = 50_000
BATCH_SIZE = 10_000
RANDOM_STATE = 42


In [ ]:
# Fit TF-IDF/NMF trên mẫu ngẫu nhiên cố định để giới hạn RAM.
base = pd.read_parquet(SCORED_PATH, columns=['id','subreddit','created_at_utc','text','sentiment_label'])
fit_df = base.sample(n=min(FIT_SAMPLE, len(base)), random_state=RANDOM_STATE)
texts = fit_df['text'].fillna('').astype(str).tolist()
vectorizer = TfidfVectorizer(max_features=30_000, min_df=5, max_df=0.90, stop_words='english', ngram_range=(1,2), strip_accents='unicode', sublinear_tf=True)
X_fit = vectorizer.fit_transform(texts)
nmf = NMF(n_components=N_TOPICS, init='nndsvda', random_state=RANDOM_STATE, max_iter=200, alpha_W=0.00005, l1_ratio=0.1)
W_fit = nmf.fit_transform(X_fit)
fit_df = fit_df.copy()
fit_df['topic_id'] = W_fit.argmax(axis=1)
print('Fit rows:', len(fit_df), '| vocabulary:', len(vectorizer.get_feature_names_out()), '| topics:', N_TOPICS)


In [ ]:
# In từ khóa nổi bật và ví dụ bài đại diện để diễn giải/đặt tên topic.
terms = vectorizer.get_feature_names_out()
topic_rows = []
for k, weights in enumerate(nmf.components_):
    top_idx = weights.argsort()[-12:][::-1]
    topic_rows.append({'topic_id': k, 'top_terms': ', '.join(terms[top_idx])})
topics = pd.DataFrame(topic_rows)
display(topics)
representatives = []
for k in range(N_TOPICS):
    idx = np.flatnonzero(fit_df['topic_id'].to_numpy() == k)
    if len(idx):
        strengths = W_fit[idx, k]
        chosen = idx[np.argsort(strengths)[-5:][::-1]]
        representatives.extend(fit_df.iloc[chosen][['topic_id','subreddit','created_at_utc','sentiment_label','text']].to_dict('records'))
rep_df = pd.DataFrame(representatives)
display(rep_df.head(30))
topics.to_csv(DATA_DIR / 'topic_terms.csv', index=False)
rep_df.to_csv(DATA_DIR / 'topic_representative_posts.csv', index=False)


In [ ]:
# Gán topic cho toàn bộ parquet theo batch, lưu kết quả để không phải tính lại.
TOPIC_PATH = DATA_DIR / 'reddit_posts_topics.parquet'
if not TOPIC_PATH.exists():
    pf = pq.ParquetFile(SCORED_PATH)
    writer = None
    for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=['id','subreddit','created_at_utc','sentiment_label','text']):
        part = batch.to_pandas()
        X = vectorizer.transform(part['text'].fillna('').astype(str))
        scores = nmf.transform(X)
        part['topic_id'] = scores.argmax(axis=1).astype('int16')
        part['topic_strength'] = scores.max(axis=1).astype('float32')
        table = __import__('pyarrow').Table.from_pandas(part.drop(columns=['text']), preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(TOPIC_PATH, table.schema, compression='zstd')
        writer.write_table(table)
    if writer is not None: writer.close()
print('Topic assignments:', TOPIC_PATH)


In [ ]:
# Tổng hợp topic theo tháng/subreddit và sentiment.
tagged = pd.read_parquet(TOPIC_PATH)
tagged['month'] = tagged['created_at_utc'].dt.strftime('%Y-%m')
monthly_topic = tagged.groupby(['month','topic_id']).size().rename('posts').reset_index()
monthly_topic.to_csv(DATA_DIR / 'topic_monthly_counts.csv', index=False)
sub_topic = tagged.groupby(['subreddit','topic_id']).size().rename('posts').reset_index()
sub_topic.to_csv(DATA_DIR / 'topic_subreddit_counts.csv', index=False)
topic_sent = tagged.groupby(['topic_id','sentiment_label'], dropna=False).size().rename('posts').reset_index()
topic_sent.to_csv(DATA_DIR / 'topic_sentiment_counts.csv', index=False)
display(monthly_topic.tail(20))
display(sub_topic.sort_values('posts', ascending=False).head(25))
print('Hoàn tất. Topic ID là nhãn máy; dùng top terms và bài mẫu để gán tên có ý nghĩa.')


## Diễn giải
- `topic_id` chỉ là cụm từ do NMF phát hiện; đặt tên sau khi xem từ khóa và bài mẫu.
- Mô hình fit trên mẫu 50.000 bài rồi áp dụng toàn bộ tập. Topic thưa/không rõ nghĩa có thể do số topic, mẫu fit hoặc tiền xử lý; nêu đây là baseline thăm dò.
- Dataset chỉ bao phủ 19 subreddit Data Science/AI, và thời gian có khoảng trống. Không diễn giải thành xu hướng toàn Reddit.
